# DEAI – Source Data Model (SDM) Pipeline

Dit notebook maakt een **Source Data Model (SDM)** op basis van de aangeleverde bestanden:

- `CRM-data.sqlite`
- `GO_SALES-data.sqlite`
- `GO_STAFF-data.sqlite`
- `INVENTORY_LEVELS-data.csv`
- `PRODUCT_FORECAST-data.csv`
- `SALES_TARGET-data.csv`

Het notebook:
1. controleert of alle bestanden bestaan;
2. leest alle tabellen uit de SQLite-bronnen;
3. leest alle CSV-bestanden;
4. maakt een nieuwe SQLite SDM-database;
5. laadt alle brondata in SDM-tabellen;
6. houdt een logbestand bij;
7. toont rij-aantallen ter controle.


## 1. Imports en paden


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import logging
import shutil


In [ ]:
# Pas BASE_DIR aan als jouw bestanden ergens anders staan.
# Als het notebook in dezelfde map staat als de databestanden, werkt Path.cwd().
BASE_DIR = Path.cwd()

SOURCE_DBS = {
    "crm": BASE_DIR / "CRM-data.sqlite",
    "sales": BASE_DIR / "GO_SALES-data.sqlite",
    "staff": BASE_DIR / "GO_STAFF-data.sqlite",
}

SOURCE_CSVS = {
    "inventory_levels": BASE_DIR / "INVENTORY_LEVELS-data.csv",
    "product_forecast": BASE_DIR / "PRODUCT_FORECAST-data.csv",
    "sales_target": BASE_DIR / "SALES_TARGET-data.csv",
}

SDM_DB = BASE_DIR / "GO_SDM.db"
LOG_DIR = BASE_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
SDM_LOG_PATH = LOG_DIR / "go_sdm_etl.log"

print("BASE_DIR:", BASE_DIR)
print("SDM_DB:", SDM_DB)
print("LOG:", SDM_LOG_PATH)


## 2. Logging instellen


In [ ]:
logger = logging.getLogger("go_sdm_etl")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

formatter = logging.Formatter(
    "%(asctime)s|%(levelname)s|%(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler = logging.FileHandler(SDM_LOG_PATH, encoding="utf-8")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.propagate = False


def log_event(level, process_step, table_name="", action="", row_count="", details=""):
    message = f"{process_step}|{table_name}|{action}|{row_count}|{details}"
    if level == "ERROR":
        logger.error(message)
    elif level == "WARNING":
        logger.warning(message)
    else:
        logger.info(message)


def reset_sdm_log():
    if logger.hasHandlers():
        for handler in list(logger.handlers):
            handler.close()
            logger.removeHandler(handler)

    if SDM_LOG_PATH.exists():
        SDM_LOG_PATH.unlink()

    new_handler = logging.FileHandler(SDM_LOG_PATH, encoding="utf-8")
    new_handler.setLevel(logging.INFO)
    new_handler.setFormatter(formatter)
    logger.addHandler(new_handler)

    print(f"Logbestand gereset: {SDM_LOG_PATH}")


## 3. Bestandcontrole


In [ ]:
def check_files():
    missing = []

    for label, path in SOURCE_DBS.items():
        if path.exists():
            log_event("INFO", "FILE_CHECK", label, "FOUND_SQLITE", "", str(path))
            print(f"SQLite gevonden: {label} -> {path.name}")
        else:
            log_event("ERROR", "FILE_CHECK", label, "MISSING_SQLITE", "", str(path))
            missing.append(path)

    for label, path in SOURCE_CSVS.items():
        if path.exists():
            log_event("INFO", "FILE_CHECK", label, "FOUND_CSV", "", str(path))
            print(f"CSV gevonden: {label} -> {path.name}")
        else:
            log_event("ERROR", "FILE_CHECK", label, "MISSING_CSV", "", str(path))
            missing.append(path)

    if missing:
        raise FileNotFoundError("Niet alle bronbestanden zijn gevonden: " + ", ".join(str(p) for p in missing))

    print("Alle bronbestanden zijn gevonden.")


check_files()


## 4. Hulpfuncties voor SQLite-tabellen


In [ ]:
def get_user_tables(conn: sqlite3.Connection) -> list[str]:
    rows = conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """).fetchall()
    return [row[0] for row in rows]


def inspect_sqlite_sources():
    overview = []

    for source_label, db_path in SOURCE_DBS.items():
        with sqlite3.connect(db_path) as conn:
            tables = get_user_tables(conn)

            for table in tables:
                count = conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
                columns = conn.execute(f'PRAGMA table_info("{table}")').fetchall()
                column_names = ", ".join([col[1] for col in columns])

                overview.append({
                    "bron": source_label,
                    "bronbestand": db_path.name,
                    "tabel": table,
                    "aantal_rijen": count,
                    "kolommen": column_names
                })

    return pd.DataFrame(overview)


sqlite_overview = inspect_sqlite_sources()
sqlite_overview


## 5. Hulpfuncties voor CSV-bestanden


In [ ]:
def read_csv_robust(path: Path) -> pd.DataFrame:
    # SALES_TARGET bevat speciale tekens. Daarom proberen we meerdere encodings.
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            log_event("INFO", "EXTRACT_CSV", path.name, "READ_CSV", len(df), f"encoding={enc}")
            return df
        except Exception as e:
            last_error = e

    log_event("ERROR", "EXTRACT_CSV", path.name, "FAILED_READ_CSV", "", str(last_error))
    raise last_error


def inspect_csv_sources():
    overview = []

    for source_label, csv_path in SOURCE_CSVS.items():
        df = read_csv_robust(csv_path)
        overview.append({
            "bron": source_label,
            "bronbestand": csv_path.name,
            "aantal_rijen": len(df),
            "aantal_kolommen": len(df.columns),
            "kolommen": ", ".join(df.columns)
        })

    return pd.DataFrame(overview)


csv_overview = inspect_csv_sources()
csv_overview


## 6. SDM resetten

Deze functie verwijdert de bestaande SDM-database en maakt een nieuwe lege SQLite-database.


In [ ]:
def reset_sdm():
    if SDM_DB.exists():
        SDM_DB.unlink()
        log_event("INFO", "RESET_SDM", "GO_SDM", "DELETE_EXISTING_DB", 1, str(SDM_DB))

    with sqlite3.connect(SDM_DB) as conn:
        conn.execute("PRAGMA foreign_keys = OFF")
        conn.commit()

    log_event("INFO", "RESET_SDM", "GO_SDM", "CREATE_EMPTY_DB", 1, str(SDM_DB))
    print(f"Nieuwe lege SDM aangemaakt: {SDM_DB}")


## 7. Data laden naar het SDM

Strategie:
- Iedere brontabel wordt 1-op-1 overgenomen.
- De SDM-tabelnaam krijgt een prefix van de bron, bijvoorbeeld:
  - `crm_customer`
  - `sales_order_header`
  - `staff_sales_representative`
  - `csv_inventory_levels`

Dit voorkomt naamconflicten tussen bronnen.


In [ ]:
def clean_table_name(name: str) -> str:
    return (
        name.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
    )


def load_sqlite_table_to_sdm(source_label: str, source_db: Path, source_table: str, target_conn: sqlite3.Connection) -> int:
    target_table = f"{source_label}_{clean_table_name(source_table)}"

    with sqlite3.connect(source_db) as source_conn:
        df = pd.read_sql_query(f'SELECT * FROM "{source_table}"', source_conn)

    df.to_sql(target_table, target_conn, if_exists="replace", index=False)

    log_event(
        "INFO",
        "LOAD_SQLITE_TO_SDM",
        target_table,
        "REPLACE_TABLE",
        len(df),
        f"source={source_db.name}; source_table={source_table}"
    )

    return len(df)


def load_csv_to_sdm(source_label: str, csv_path: Path, target_conn: sqlite3.Connection) -> int:
    target_table = f"csv_{clean_table_name(source_label)}"

    df = read_csv_robust(csv_path)
    df.to_sql(target_table, target_conn, if_exists="replace", index=False)

    log_event(
        "INFO",
        "LOAD_CSV_TO_SDM",
        target_table,
        "REPLACE_TABLE",
        len(df),
        f"source={csv_path.name}"
    )

    return len(df)


def load_all_sources_to_sdm() -> pd.DataFrame:
    results = []

    log_event("INFO", "RUN_SDM_PIPELINE", "", "START", "", "Start laden bronnen naar SDM")

    with sqlite3.connect(SDM_DB) as target_conn:
        target_conn.execute("PRAGMA foreign_keys = OFF")

        # SQLite bronnen laden
        for source_label, db_path in SOURCE_DBS.items():
            with sqlite3.connect(db_path) as source_conn:
                tables = get_user_tables(source_conn)

            for table in tables:
                loaded_rows = load_sqlite_table_to_sdm(source_label, db_path, table, target_conn)
                results.append({
                    "bron_type": "sqlite",
                    "bron": source_label,
                    "bronbestand": db_path.name,
                    "bron_tabel": table,
                    "sdm_tabel": f"{source_label}_{clean_table_name(table)}",
                    "geladen_rijen": loaded_rows
                })

        # CSV bronnen laden
        for source_label, csv_path in SOURCE_CSVS.items():
            loaded_rows = load_csv_to_sdm(source_label, csv_path, target_conn)
            results.append({
                "bron_type": "csv",
                "bron": source_label,
                "bronbestand": csv_path.name,
                "bron_tabel": csv_path.name,
                "sdm_tabel": f"csv_{clean_table_name(source_label)}",
                "geladen_rijen": loaded_rows
            })

        target_conn.commit()
        target_conn.execute("PRAGMA foreign_keys = ON")

    total_rows = sum(row["geladen_rijen"] for row in results)
    log_event("INFO", "RUN_SDM_PIPELINE", "", "END", total_rows, "Alle bronnen naar SDM geladen")

    return pd.DataFrame(results)


## 8. Controles op het SDM


In [ ]:
def sdm_tables() -> list[str]:
    with sqlite3.connect(SDM_DB) as conn:
        return get_user_tables(conn)


def row_counts_sdm() -> pd.DataFrame:
    data = []

    with sqlite3.connect(SDM_DB) as conn:
        tables = get_user_tables(conn)

        for table in tables:
            count = conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
            data.append({
                "sdm_tabel": table,
                "aantal_rijen": count
            })
            log_event("INFO", "ROW_COUNT", table, "COUNT_ROWS", count, "Rijtelling in SDM")

    return pd.DataFrame(data).sort_values("sdm_tabel").reset_index(drop=True)


def preview_table(table_name: str, limit: int = 5) -> pd.DataFrame:
    with sqlite3.connect(SDM_DB) as conn:
        return pd.read_sql_query(f'SELECT * FROM "{table_name}" LIMIT {limit}', conn)


## 9. Pipeline uitvoeren


In [ ]:
# Gebruik dit blok om opnieuw te beginnen met een schoon SDM en een schone log.
reset_sdm_log()
check_files()
reset_sdm()

load_results = load_all_sources_to_sdm()
load_results


In [ ]:
counts = row_counts_sdm()
counts


## 10. Voorbeeld: tabel bekijken


In [ ]:
# Toon beschikbare SDM-tabellen
sdm_tables()


In [ ]:
# Pas eventueel de tabelnaam aan om een andere tabel te bekijken.
preview_table("sales_order_header", limit=5)


## 11. Assessment-uitleg

In dit notebook is een Source Data Model gemaakt waarin alle brondata 1-op-1 wordt ingeladen.  
De data komt uit meerdere operationele bronnen: SQLite-databases en CSV-bestanden.  
Om naamconflicten te voorkomen krijgen de SDM-tabellen een prefix op basis van de bron.

Voorbeeld:
- `CRM-data.sqlite` → tabellen met prefix `crm_`
- `GO_SALES-data.sqlite` → tabellen met prefix `sales_`
- `GO_STAFF-data.sqlite` → tabellen met prefix `staff_`
- CSV-bestanden → tabellen met prefix `csv_`

Daarnaast wordt logging toegepast zodat zichtbaar is:
- welke bestanden zijn gevonden;
- welke tabellen zijn ingelezen;
- hoeveel rijen per tabel zijn geladen;
- wanneer de pipeline is gestart en afgerond.
